# AI Prep Search via Foundation Models

This notebook builds and deploys an MLflow pyfunc model that recreates the `ai_prep_search` functionality using Databricks Foundation Model APIs. It accepts the structured JSON output from `ai_parse_document` (or plain text/markdown), performs semantic chunking, enriches each chunk with document context via an LLM, and produces output compatible with Databricks AI Search indexes.

The foundation model is configurable via the `CHUNKING_MODEL` variable below.

In [0]:
# No heavy ML packages — the foundation model is called remotely via FMAPI.
%pip install -U \
  "mlflow>=3.6.0" \
  "openai>=1.37.0,<4.0.0"
%restart_python

In [0]:
# --- Workspace configuration ---
# Update CATALOG and SCHEMA to match your Unity Catalog location.
CATALOG = "justinm_demo"
SCHEMA = "parse"

# Foundation model for chunking enrichment (context sentence, table summaries,
# related questions). Change this to any FMAPI-compatible model.
CHUNKING_MODEL = "databricks-gemini-3-8-flash"

REGISTERED_MODEL_NAME = f"{CATALOG}.{SCHEMA}.ai-prep-search-fmapi"
ENDPOINT_NAME = "ai-prep-search-fmapi"

# Chunking parameters
MAX_CHUNK_CHARS = 2000   # Approximate max characters per chunk
LLM_MAX_TOKENS = 2048    # Max tokens for enrichment LLM calls

print({
    "chunking_model": CHUNKING_MODEL,
    "registered_model": REGISTERED_MODEL_NAME,
    "endpoint": ENDPOINT_NAME,
})

In [0]:
import json
import time
import uuid

import mlflow
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from mlflow import MlflowClient
from mlflow.models import ModelSignature
from mlflow.models.resources import DatabricksServingEndpoint
from mlflow.types.schema import ColSpec, DataType, Schema

mlflow.set_registry_uri("databricks-uc")
print("Imports loaded")

In [0]:
class AiPrepSearchModel(mlflow.pyfunc.PythonModel):
    """Recreates ai_prep_search using Databricks Foundation Model APIs.

    Accepts structured ai_parse_document 2.0 JSON or plain text/markdown,
    splits into semantic chunks, enriches with document context via an LLM,
    and returns search-ready output matching the ai_prep_search schema.
    """

    _CONTEXT_PROMPT = (
        "You are a document analysis assistant. Given the following document "
        "text, write a single sentence that describes the document's main "
        "topic and purpose. This sentence will provide context for search "
        "indexing. Return ONLY the context sentence, nothing else.\n\n"
        "Document text (first 3000 characters):\n{text}"
    )

    _TABLE_SUMMARY_PROMPT = (
        "You are a data analysis assistant. Given the following table from a "
        "document, provide:\n"
        "1. A brief summary of what the table contains (1-2 sentences)\n"
        "2. 3-5 natural language questions this table could answer\n\n"
        "Return ONLY a JSON object: "
        '{{"summary": "...", "questions": ["...", "..."]}}\n\n'
        "Table content:\n{table_content}"
    )

    def __init__(self, chunking_model=CHUNKING_MODEL,
                 max_chunk_chars=MAX_CHUNK_CHARS,
                 llm_max_tokens=LLM_MAX_TOKENS):
        self.chunking_model = chunking_model
        self.max_chunk_chars = max_chunk_chars
        self.llm_max_tokens = llm_max_tokens

    def load_context(self, context):
        import logging
        self.log = logging.getLogger("ai_prep_search_fmapi")
        self._ws = WorkspaceClient()
        self.client = openai.OpenAI(
            api_key=self._get_token(),
            base_url=f"{self._ws.config.host}/serving-endpoints",
        )

    def _get_token(self):
        """Fetch a fresh bearer token. In Model Serving, DATABRICKS_TOKEN is
        auto-rotated by the serving infra; fall back to WorkspaceClient auth
        in notebook context."""
        import os
        token = os.environ.get("DATABRICKS_TOKEN") or self._ws.config.token
        if not token:
            headers = self._ws.config.authenticate()
            token = headers.get("Authorization", "").removeprefix("Bearer ").strip()
        return token

    # ── LLM helpers ──────────────────────────────────────────────────

    def _llm_call(self, prompt, max_tokens=None):
        """Single LLM call; returns text or empty string on failure."""
        try:
            resp = self.client.chat.completions.create(
                model=self.chunking_model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens or self.llm_max_tokens,
            )
            content = resp.choices[0].message.content
            if isinstance(content, list):
                return "".join(
                    p.get("text", "") if isinstance(p, dict)
                    else getattr(p, "text", "")
                    for p in content
                ).strip()
            return (content or "").strip()
        except Exception as exc:
            self.log.warning("LLM call failed: %s", exc)
            if not hasattr(self, "_llm_errors"):
                self._llm_errors = []
            self._llm_errors.append(f"llm_call_failed: {type(exc).__name__}: {exc}"[:500])
            return ""

    def _generate_context_sentence(self, full_text):
        """Generate a one-line document context sentence."""
        return self._llm_call(
            self._CONTEXT_PROMPT.format(text=full_text[:3000]),
            max_tokens=256,
        )

    def _generate_table_enrichment(self, table_html):
        """Generate table summary and related questions."""
        raw = self._llm_call(
            self._TABLE_SUMMARY_PROMPT.format(table_content=table_html[:3000]),
            max_tokens=512,
        )
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
            if raw.endswith("```"):
                raw = raw[:-3].strip()
        try:
            result = json.loads(raw)
            return result.get("summary", ""), result.get("questions", [])
        except (json.JSONDecodeError, AttributeError):
            return "", []

    _METADATA_ENRICHMENT_PROMPT = (
        "You are a document analysis assistant. Given the following document "
        "content, extract or infer metadata for search indexing.\n\n"
        "Document text (first 4000 characters):\n{text}\n\n"
        "Source file: {source_uri}\n"
        "Number of content sections: {n_chunks}\n\n"
        "Return a JSON object with these fields:\n"
        '{{\n'
        '  "document_title": "The document\'s main title or a concise descriptive name",\n'
        '  "sections": ["A short descriptive section/topic label for each of the {n_chunks} content sections, in document order"],\n'
        '  "page_header_default": "A reasonable default page header (e.g., organization name or document title abbreviation)",\n'
        '  "page_footer_default": "A reasonable default page footer (e.g., source, date, or document classification)"\n'
        '}}\n\n'
        "Return ONLY the JSON object."
    )

    def _generate_metadata_enrichment(self, full_text, source_uri, n_chunks, doc_ctx):
        """Generate enriched metadata for the document in a single LLM call."""
        has_title = bool(doc_ctx.get("title", "").strip())
        has_sections = len(doc_ctx.get("section_headers", [])) >= max(1, n_chunks // 2)
        has_headers = bool(doc_ctx.get("page_headers"))
        has_footers = bool(doc_ctx.get("page_footers"))

        if has_title and has_sections and has_headers and has_footers:
            return {
                "document_title": doc_ctx["title"],
                "sections": [sh["text"] for sh in doc_ctx["section_headers"]],
                "page_header_default": doc_ctx["title"],
                "page_footer_default": next(iter(doc_ctx["page_footers"].values()), ""),
            }

        raw = self._llm_call(
            self._METADATA_ENRICHMENT_PROMPT.format(
                text=full_text[:4000],
                source_uri=source_uri or "(unknown)",
                n_chunks=n_chunks,
            ),
            max_tokens=512,
        )
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
            if raw.endswith("```"):
                raw = raw[:-3].strip()
        try:
            result = json.loads(raw)
        except (json.JSONDecodeError, AttributeError):
            result = {}

        title = doc_ctx.get("title") or result.get("document_title", "")
        filename = source_uri.rsplit("/", 1)[-1] if source_uri else ""
        return {
            "document_title": title,
            "sections": result.get("sections", []),
            "page_header_default": result.get("page_header_default", title),
            "page_footer_default": result.get("page_footer_default", filename),
        }

    # ── Input parsing ────────────────────────────────────────────────

    def _parse_input(self, parsed_doc_json):
        """Detect whether input is ai_parse_document 2.0 JSON or plain text.

        Returns (elements_or_text, pages, metadata_dict, is_plain_text).
        """
        if not parsed_doc_json or (
            isinstance(parsed_doc_json, float) and pd.isna(parsed_doc_json)
        ):
            return "", None, None, True
        text = str(parsed_doc_json).strip()
        try:
            data = json.loads(text)
            if isinstance(data, dict) and "document" in data:
                doc = data["document"]
                return (
                    doc.get("elements", []),
                    doc.get("pages", []),
                    data.get("metadata", {}),
                    False,
                )
        except (json.JSONDecodeError, TypeError):
            pass
        return text, None, None, True

    # ── Document context extraction ──────────────────────────────────

    def _extract_document_context(self, elements):
        """Pull document-level metadata from typed elements."""
        ctx = {
            "title": "",
            "section_headers": [],
            "page_headers": {},
            "page_footers": {},
            "page_numbers": {},
            "captions": [],
            "footnotes": [],
        }
        for el in elements:
            t = el.get("type", "text")
            c = el.get("content") or ""
            pids = [b.get("page_id", 0) for b in (el.get("bbox") or [])]
            pid = pids[0] if pids else 0
            eid = el.get("id", 0)
            if t == "title" and not ctx["title"]:
                ctx["title"] = c
            elif t == "section_header":
                ctx["section_headers"].append({"text": c, "element_id": eid})
            elif t == "page_header":
                ctx["page_headers"][pid] = c
            elif t == "page_footer":
                ctx["page_footers"][pid] = c
            elif t == "page_number":
                ctx["page_numbers"][pid] = c
            elif t == "caption":
                ctx["captions"].append({"text": c, "page_id": pid})
            elif t == "footnote":
                ctx["footnotes"].append({"text": c, "page_id": pid})
        return ctx

    # ── Semantic chunking ────────────────────────────────────────────

    def _semantic_chunk_elements(self, elements):
        """Group elements into semantic chunks by type and size."""
        BOUNDARY = {"title", "section_header"}
        STANDALONE = {"table", "figure"}
        SKIP = {"page_header", "page_footer", "page_number"}

        chunks, cur_els, cur_len = [], [], 0

        def flush():
            nonlocal cur_els, cur_len
            if cur_els:
                chunks.append(cur_els[:])
            cur_els, cur_len = [], 0

        for el in elements:
            t = el.get("type", "text")
            if t in SKIP:
                continue
            c = el.get("content") or el.get("description") or ""

            if t in STANDALONE:
                flush()
                chunks.append([el])
                continue
            if t in BOUNDARY and cur_els:
                flush()
            if cur_len and cur_len + len(c) > self.max_chunk_chars:
                flush()

            cur_els.append(el)
            cur_len += len(c)

        flush()
        return chunks

    def _chunk_plain_text(self, text):
        """Chunk plain text by paragraphs respecting size limits."""
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        if not paragraphs:
            return [text[:self.max_chunk_chars]] if text.strip() else [""]

        chunks, cur = [], ""
        for para in paragraphs:
            if cur and len(cur) + len(para) + 2 > self.max_chunk_chars:
                chunks.append(cur)
                cur = para
            else:
                cur = f"{cur}\n\n{para}" if cur else para
        if cur:
            chunks.append(cur)
        return chunks

    # ── chunk_to_embed construction ──────────────────────────────────

    def _get_chunk_context(self, chunk_elements, doc_ctx):
        """Resolve nearest page header/footer/section/caption/footnote."""
        page_ids = set()
        for el in chunk_elements:
            for b in (el.get("bbox") or []):
                page_ids.add(b.get("page_id", 0))
        primary = min(page_ids) if page_ids else 0

        min_eid = min(
            (el.get("id", float("inf")) for el in chunk_elements),
            default=float("inf"),
        )
        section = ""
        for sh in doc_ctx.get("section_headers", []):
            if sh["element_id"] <= min_eid:
                section = sh["text"]

        caption, footnote = "", ""
        for cap in doc_ctx.get("captions", []):
            if cap["page_id"] in page_ids:
                caption = cap["text"]
        for fn in doc_ctx.get("footnotes", []):
            if fn["page_id"] in page_ids:
                footnote = fn["text"]

        return {
            "page_header": doc_ctx.get("page_headers", {}).get(primary, ""),
            "page_footer": doc_ctx.get("page_footers", {}).get(primary, ""),
            "section_header": section,
            "caption": caption,
            "footnote": footnote,
            "page_number": str(doc_ctx.get("page_numbers", {}).get(primary, "")) or str(primary + 1),
            "page_ids": sorted(page_ids),
        }

    @staticmethod
    def _build_chunk_to_embed(chunk_text, doc_ctx, chunk_ctx,
                               is_table=False, table_summary="",
                               related_questions=None):
        """Assemble the enriched chunk_to_embed following the ai_prep_search
        template: fixed metadata labels are always rendered; optional items
        (context sentence, table summary, related questions) appear only when
        available."""
        fields = [
            ("Document Title", doc_ctx.get('title', '')),
            ("Page Header", chunk_ctx.get('page_header', '')),
            ("Page Footer", chunk_ctx.get('page_footer', '')),
            ("Section Header", chunk_ctx.get('section_header', '')),
            ("Caption", chunk_ctx.get('caption', '')),
            ("Footnote", chunk_ctx.get('footnote', '')),
            ("Page Number", chunk_ctx.get('page_number', '')),
        ]
        lines = [f"{label}: {value}" for label, value in fields if value]
        ctx_sentence = doc_ctx.get("context_sentence", "")
        if ctx_sentence:
            lines.append(ctx_sentence)
        if is_table and table_summary:
            lines.append(f"Table summary: {table_summary}")
        lines.append(f"Content:\n{chunk_text}")
        if is_table and related_questions:
            lines.append(
                "Related questions:\n"
                + "\n".join(f"- {q}" for q in related_questions)
            )
        return "\n".join(lines)

    # ── Row handlers ─────────────────────────────────────────────────

    def _handle_structured(self, elements, pages, source_uri, doc_id):
        """Process ai_parse_document 2.0 structured input."""
        doc_ctx = self._extract_document_context(elements)

        full_text = "\n".join(
            el.get("content") or el.get("description") or ""
            for el in elements
            if el.get("type") not in {"page_header", "page_footer", "page_number"}
        )
        doc_ctx["context_sentence"] = (
            self._generate_context_sentence(full_text) if full_text.strip() else ""
        )

        element_chunks = self._semantic_chunk_elements(elements)

        # Enrich missing document metadata with a single LLM call
        enriched = self._generate_metadata_enrichment(
            full_text, source_uri, len(element_chunks), doc_ctx
        )
        if not doc_ctx["title"]:
            doc_ctx["title"] = enriched.get("document_title", "")

        output_chunks = []
        for pos, chunk_els in enumerate(element_chunks):
            texts, is_table, is_figure = [], False, False
            for el in chunk_els:
                t = el.get("type", "text")
                if t == "table":
                    is_table = True
                    texts.append(el.get("content") or "")
                elif t == "figure":
                    is_figure = True
                    texts.append(f'[Figure: {el.get("description") or ""}]')
                else:
                    texts.append(el.get("content") or "")

            chunk_to_retrieve = "\n".join(texts)
            if not chunk_to_retrieve.strip():
                continue

            chunk_ctx = self._get_chunk_context(chunk_els, doc_ctx)

            # ── Fill missing fields from LLM-enriched metadata ───────
            sections = enriched.get("sections", [])
            if not chunk_ctx["section_header"] and pos < len(sections):
                chunk_ctx["section_header"] = sections[pos]
            if not chunk_ctx["page_header"]:
                chunk_ctx["page_header"] = enriched.get(
                    "page_header_default", doc_ctx.get("title", "")
                )
            if not chunk_ctx["page_footer"]:
                chunk_ctx["page_footer"] = enriched.get("page_footer_default", "")

            table_summary, related_qs = "", []
            if is_table:
                table_summary, related_qs = self._generate_table_enrichment(
                    chunk_to_retrieve
                )
                if not chunk_ctx["caption"] and table_summary:
                    chunk_ctx["caption"] = table_summary
            if is_figure and not chunk_ctx["caption"]:
                fig_descs = [
                    el.get("description", "") for el in chunk_els
                    if el.get("type") == "figure" and el.get("description")
                ]
                if fig_descs:
                    chunk_ctx["caption"] = fig_descs[0]

            chunk_to_embed = self._build_chunk_to_embed(
                chunk_to_retrieve, doc_ctx, chunk_ctx,
                is_table=is_table, table_summary=table_summary,
                related_questions=related_qs,
            )

            chunk_pages = []
            for pid in chunk_ctx["page_ids"]:
                ref = {"page_id": pid, "image_uri": ""}
                for p in (pages or []):
                    if p.get("id") == pid:
                        ref["image_uri"] = p.get("image_uri", "")
                        break
                chunk_pages.append(ref)

            output_chunks.append({
                "chunk_id": f"{doc_id}_{pos}",
                "chunk_position": pos,
                "chunk_to_retrieve": chunk_to_retrieve,
                "chunk_to_embed": chunk_to_embed,
                "metadata": {},
                "pages": chunk_pages,
            })

        return {
            "document": {
                "contents": output_chunks,
                "pages": pages or [],
                "source_uri": source_uri,
            },
            "error_status": [],
        }

    def _handle_plain_text(self, text, source_uri, doc_id):
        """Process plain text or markdown input."""
        ctx_sentence = (
            self._generate_context_sentence(text) if text.strip() else ""
        )
        doc_ctx = {"title": "", "context_sentence": ctx_sentence}
        empty_ctx = {
            "page_header": "", "page_footer": "", "section_header": "",
            "caption": "", "footnote": "", "page_number": "",
        }

        chunks = self._chunk_plain_text(text)
        output_chunks = []
        for pos, chunk_text in enumerate(chunks):
            output_chunks.append({
                "chunk_id": f"{doc_id}_{pos}",
                "chunk_position": pos,
                "chunk_to_retrieve": chunk_text,
                "chunk_to_embed": self._build_chunk_to_embed(
                    chunk_text, doc_ctx, empty_ctx,
                ),
                "metadata": {},
                "pages": [],
            })

        return {
            "document": {
                "contents": output_chunks,
                "pages": [],
                "source_uri": source_uri,
            },
            "error_status": [],
        }

    def _handle_row(self, row):
        parsed_json = row.get("parsed_doc_json", "")
        source_uri = row.get("source_uri") or ""
        if isinstance(source_uri, float) and pd.isna(source_uri):
            source_uri = ""
        doc_id = str(uuid.uuid4())

        elements, pages, _meta, is_plain = self._parse_input(parsed_json)

        self._llm_errors = []
        if is_plain:
            out = self._handle_plain_text(elements, source_uri, doc_id)
        else:
            out = self._handle_structured(elements, pages, source_uri, doc_id)
        # Surface LLM enrichment failures instead of silently returning blanks.
        out["error_status"] = out.get("error_status", []) + self._llm_errors
        return json.dumps(out, ensure_ascii=False)

    def predict(self, context, model_input, params=None):
        # Refresh token on every scoring call for long-lived replicas.
        self.client.api_key = self._get_token()

        if isinstance(model_input, pd.DataFrame):
            rows = model_input.to_dict(orient="records")
        elif isinstance(model_input, dict):
            rows = [model_input]
        else:
            rows = list(model_input)

        output = []
        for row in rows:
            try:
                output.append(self._handle_row(row))
            except Exception as exc:
                self.log.exception("Row failed")
                src = row.get("source_uri", "") if isinstance(row, dict) else ""
                output.append(json.dumps({
                    "document": {
                        "contents": [], "pages": [],
                        "source_uri": src,
                    },
                    "error_status": [f"processing_error: {exc}"],
                }, ensure_ascii=False))

        return pd.DataFrame({"response": output})

print(f"AiPrepSearchModel defined (chunking_model={CHUNKING_MODEL})")

In [0]:
# Build a sample ai_parse_document 2.0 envelope for the input example.
_SAMPLE_ENVELOPE = json.dumps({
    "document": {
        "pages": [{"id": 0, "image_uri": "sample.pdf"}],
        "elements": [
            {"id": 0, "type": "title", "content": "Sample Report",
             "confidence": 0.95,
             "bbox": [{"coord": [100, 50, 500, 80], "page_id": 0}],
             "description": None},
            {"id": 1, "type": "text",
             "content": "This is a sample document for testing.",
             "confidence": 0.90,
             "bbox": [{"coord": [100, 100, 500, 200], "page_id": 0}],
             "description": None},
        ],
    },
    "error_status": [],
    "metadata": {
        "id": "sample-001", "version": "2.0",
        "file_metadata": {
            "file_path": "sample.pdf", "file_name": "sample.pdf",
            "file_size": 1024,
            "file_modification_time": "1970-01-01T00:00:00Z",
        },
    },
})

signature = ModelSignature(
    inputs=Schema([
        ColSpec(DataType.string, "parsed_doc_json", required=True),
        ColSpec(DataType.string, "source_uri", required=False),
    ]),
    outputs=Schema([ColSpec(DataType.string, "response", required=True)]),
)

input_example = pd.DataFrame([{
    "parsed_doc_json": _SAMPLE_ENVELOPE,
    "source_uri": "/Volumes/catalog/schema/volume/sample.pdf",
}])

with mlflow.start_run(run_name="ai_prep_search_fmapi") as run:
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=AiPrepSearchModel(),
        signature=signature,
        input_example=input_example,
        resources=[
            DatabricksServingEndpoint(endpoint_name=CHUNKING_MODEL),
        ],
        pip_requirements=[
            "--extra-index-url=https://pypi.org/simple",
            "mlflow>=3.6.0",
            "openai>=1.37.0,<4.0.0",
            "pandas>=2.2.0",
            "numpy>=1.26.0,<3.0.0",
            "databricks-sdk>=0.62.0",
            "cloudpickle>=3.1.0",
        ],
        metadata={
            "chunking_model": CHUNKING_MODEL,
            "output_schema": "ai_prep_search/1.0",
        },
    )
    logged_run_id = run.info.run_id
    logged_model_uri = model_info.model_uri

# Register and alias
client = MlflowClient(registry_uri="databricks-uc")
try:
    client.get_registered_model(REGISTERED_MODEL_NAME)
except Exception:
    client.create_registered_model(REGISTERED_MODEL_NAME)

new_version = client.create_model_version(
    name=REGISTERED_MODEL_NAME,
    source=logged_model_uri,
    run_id=logged_run_id,
)
latest_model_version = new_version.version
client.set_registered_model_alias(
    name=REGISTERED_MODEL_NAME, alias="Production",
    version=latest_model_version,
)
print({
    "run_id": logged_run_id,
    "model_uri": f"models:/{REGISTERED_MODEL_NAME}/{latest_model_version}",
    "alias": "Production",
})

In [0]:
# Create or update a CPU serving endpoint. No GPU needed — all inference
# is done via remote API calls to the foundation model.
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

workspace = WorkspaceClient()
served_entity = ServedEntityInput(
    name=ENDPOINT_NAME,
    entity_name=REGISTERED_MODEL_NAME,
    entity_version=latest_model_version,
    workload_size="Small",
    scale_to_zero_enabled=False,
)

try:
    workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    workspace.serving_endpoints.update_config(
        name=ENDPOINT_NAME, served_entities=[served_entity]
    )
    action = "updated"
except NotFound:
    workspace.serving_endpoints.create(
        name=ENDPOINT_NAME,
        config=EndpointCoreConfigInput(name=ENDPOINT_NAME, served_entities=[served_entity]),
    )
    action = "created"

# Poll until ready
deadline = time.time() + 60 * 60
while time.time() < deadline:
    endpoint = workspace.serving_endpoints.get(name=ENDPOINT_NAME)
    ready = str(getattr(endpoint.state, "ready", None))
    updating = str(getattr(endpoint.state, "config_update", None))
    print({"ready": ready, "config_update": updating})
    if "READY" in ready and (
        "NOT_UPDATING" in updating or updating in {"None", "READY"}
    ):
        break
    if "FAILED" in updating:
        raise RuntimeError(f"Endpoint deployment failed: {endpoint.state}")
    time.sleep(30)
else:
    raise TimeoutError(f"Timed out waiting for {ENDPOINT_NAME}")

print({"action": action, "endpoint": ENDPOINT_NAME,
       "version": latest_model_version})

## Smoke Tests & Result Parsing

The cells below load the registered model and exercise it against several input types to verify consistent output structure, then demonstrate how to flatten the JSON envelope into the target table schema:

| Column | Type |
| --- | --- |
| chunk_id | STRING |
| chunk_position | INT |
| chunk_to_retrieve | STRING |
| chunk_to_embed | STRING |
| metadata | VARIANT |
| source_uri | STRING |

In [0]:
# ── Smoke Tests ──────────────────────────────────────────────────────
# Load the registered model and run it against varied inputs to verify
# consistent output structure.

model = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL_NAME}@Production")

# ── Test 1: Rich structured document (multi-element, multi-page) ─────
rich_doc = json.dumps({
    "document": {
        "pages": [
            {"id": 0, "image_uri": "page0.png"},
            {"id": 1, "image_uri": "page1.png"},
        ],
        "elements": [
            {"id": 0, "type": "title",
             "content": "Quarterly Financial Report Q3 2026",
             "confidence": 0.98,
             "bbox": [{"coord": [50, 30, 500, 70], "page_id": 0}],
             "description": None},
            {"id": 1, "type": "section_header",
             "content": "Executive Summary",
             "confidence": 0.95,
             "bbox": [{"coord": [50, 90, 400, 110], "page_id": 0}],
             "description": None},
            {"id": 2, "type": "text",
             "content": "Revenue grew 20% year-over-year to $4.2 billion, "
                        "driven by strong cloud adoption across enterprise "
                        "customers. Operating margins improved to 32%.",
             "confidence": 0.92,
             "bbox": [{"coord": [50, 120, 550, 250], "page_id": 0}],
             "description": None},
            {"id": 3, "type": "text",
             "content": "Customer retention rate remained above 95%, with "
                        "net revenue retention reaching 130%.",
             "confidence": 0.91,
             "bbox": [{"coord": [50, 260, 550, 350], "page_id": 0}],
             "description": None},
            {"id": 4, "type": "section_header",
             "content": "Revenue Breakdown",
             "confidence": 0.96,
             "bbox": [{"coord": [50, 30, 400, 60], "page_id": 1}],
             "description": None},
            {"id": 5, "type": "table",
             "content": "<table><tr><th>Segment</th><th>Revenue ($M)</th>"
                        "<th>YoY Growth</th></tr><tr><td>Cloud</td>"
                        "<td>2,800</td><td>+28%</td></tr><tr><td>On-Prem</td>"
                        "<td>1,400</td><td>+8%</td></tr></table>",
             "confidence": 0.88,
             "bbox": [{"coord": [50, 70, 550, 250], "page_id": 1}],
             "description": None},
            {"id": 6, "type": "page_header",
             "content": "Confidential",
             "confidence": 0.90,
             "bbox": [{"coord": [50, 5, 200, 20], "page_id": 0}],
             "description": None},
            {"id": 7, "type": "page_footer",
             "content": "\u00a9 2026 Acme Corp",
             "confidence": 0.89,
             "bbox": [{"coord": [50, 780, 300, 790], "page_id": 0}],
             "description": None},
            {"id": 8, "type": "page_number", "content": "1",
             "confidence": 0.99,
             "bbox": [{"coord": [530, 780, 550, 790], "page_id": 0}],
             "description": None},
            {"id": 9, "type": "caption",
             "content": "Table 1: Revenue by segment",
             "confidence": 0.87,
             "bbox": [{"coord": [50, 255, 400, 270], "page_id": 1}],
             "description": None},
            {"id": 10, "type": "figure", "content": None,
             "confidence": 0.85,
             "bbox": [{"coord": [50, 300, 550, 500], "page_id": 1}],
             "description": "Bar chart showing quarterly revenue trend "
                            "from Q1 2025 to Q3 2026"},
        ],
    },
    "error_status": [],
    "metadata": {
        "id": "test-doc-001", "version": "2.0",
        "file_metadata": {
            "file_path": "/Volumes/catalog/schema/vol/report.pdf",
            "file_name": "report.pdf", "file_size": 245000,
            "file_modification_time": "2026-10-01T12:00:00Z",
        },
    },
})

result1 = model.predict(pd.DataFrame([{
    "parsed_doc_json": rich_doc,
    "source_uri": "/Volumes/justinm_demo/parse/docs/report.pdf",
}]))
out1 = json.loads(result1["response"].iloc[0])

print("\u2550"*3, "Test 1: Rich structured document", "\u2550"*3)
print(f"  Chunks produced: {len(out1['document']['contents'])}")
print(f"  Error status:    {out1['error_status']}")
for i, chunk in enumerate(out1["document"]["contents"]):
    embed = chunk["chunk_to_embed"]
    assert "Document Title:" in embed, f"Chunk {i}: missing Document Title label"
    assert "Content:" in embed, f"Chunk {i}: missing Content label"
    print(f"  Chunk {i}: pos={chunk['chunk_position']}, "
          f"retrieve={len(chunk['chunk_to_retrieve'])}ch, "
          f"embed={len(embed)}ch, pages={chunk['pages']}")
print("  \u2713 All chunks have correct enrichment template")

# ── Test 2: Plain text input ─────────────────────────────────────────
result2 = model.predict(pd.DataFrame([{
    "parsed_doc_json": (
        "Databricks Lakehouse Platform unifies data warehousing and "
        "data lakes into a single architecture.\n\n"
        "It provides ACID transactions, schema enforcement, and "
        "governance across all data assets.\n\n"
        "The platform supports SQL analytics, machine learning, and "
        "real-time streaming on a single copy of data."
    ),
    "source_uri": "inline-text",
}]))
out2 = json.loads(result2["response"].iloc[0])

print(f"\n\u2550"*3, "Test 2: Plain text input", "\u2550"*3)
print(f"  Chunks produced: {len(out2['document']['contents'])}")
for chunk in out2["document"]["contents"]:
    assert chunk["chunk_to_retrieve"], "chunk_to_retrieve should not be empty"
    assert "Content:" in chunk["chunk_to_embed"]
print("  \u2713 All plain text chunks valid")

# ── Test 3: Minimal single-element document ──────────────────────────
minimal_doc = json.dumps({
    "document": {
        "pages": [{"id": 0, "image_uri": ""}],
        "elements": [
            {"id": 0, "type": "text", "content": "Hello world.",
             "confidence": 0.99,
             "bbox": [{"coord": [0, 0, 100, 20], "page_id": 0}],
             "description": None},
        ],
    },
    "error_status": [],
    "metadata": {"id": "min-001", "version": "2.0",
                  "file_metadata": {"file_path": "min.pdf",
                                     "file_name": "min.pdf",
                                     "file_size": 100,
                                     "file_modification_time": "2026-01-01T00:00:00Z"}},
})

result3 = model.predict(pd.DataFrame([{
    "parsed_doc_json": minimal_doc, "source_uri": "min.pdf",
}]))
out3 = json.loads(result3["response"].iloc[0])

print(f"\n\u2550"*3, "Test 3: Minimal document", "\u2550"*3)
assert len(out3["document"]["contents"]) == 1, \
    f"Expected 1 chunk, got {len(out3['document']['contents'])}"
assert out3["document"]["contents"][0]["chunk_to_retrieve"] == "Hello world."
print("  \u2713 Single-element document \u2192 1 chunk")

# ── Test 4: Empty / null input handling ──────────────────────────────
result4 = model.predict(pd.DataFrame([
    {"parsed_doc_json": "", "source_uri": ""},
    {"parsed_doc_json": None, "source_uri": None},
]))
for idx, resp in enumerate(result4["response"]):
    out = json.loads(resp)
    assert "document" in out, f"Row {idx}: missing 'document'"
    assert "error_status" in out, f"Row {idx}: missing 'error_status'"

print(f"\n\u2550"*3, "Test 4: Empty / null inputs", "\u2550"*3)
print("  \u2713 Both return valid envelope")

# ── Test 5: Batch consistency (3 rows, mixed types) ──────────────────
batch_input = pd.DataFrame([
    {"parsed_doc_json": rich_doc, "source_uri": "batch_1.pdf"},
    {"parsed_doc_json": "Simple text for batch test.",
     "source_uri": "batch_2.txt"},
    {"parsed_doc_json": minimal_doc, "source_uri": "batch_3.pdf"},
])
result5 = model.predict(batch_input)
assert len(result5) == 3, f"Expected 3 rows, got {len(result5)}"
REQUIRED_FIELDS = [
    "chunk_id", "chunk_position", "chunk_to_retrieve",
    "chunk_to_embed", "metadata", "pages",
]
for idx, resp in enumerate(result5["response"]):
    out = json.loads(resp)
    assert out["document"]["contents"], f"Row {idx}: no chunks"
    for chunk in out["document"]["contents"]:
        for field in REQUIRED_FIELDS:
            assert field in chunk, f"Row {idx}: missing '{field}'"

print(f"\n\u2550"*3, "Test 5: Batch consistency (3 rows)", "\u2550"*3)
print("  \u2713 All rows produce valid output with all required fields")

print("\n\u2705 All smoke tests passed!")

In [0]:
# ── Flatten JSON response into the target table schema (Pandas) ──────
# Use this pattern for quick local inspection of model output.

def flatten_prep_search_response(result_df):
    """Flatten the model's response DataFrame into chunk-level rows."""
    all_rows = []
    for _, row in result_df.iterrows():
        envelope = json.loads(row["response"])
        source_uri = envelope["document"]["source_uri"]
        for chunk in envelope["document"]["contents"]:
            all_rows.append({
                "chunk_id": chunk["chunk_id"],
                "chunk_position": chunk["chunk_position"],
                "chunk_to_retrieve": chunk["chunk_to_retrieve"],
                "chunk_to_embed": chunk["chunk_to_embed"],
                "metadata": json.dumps(chunk["metadata"]),
                "source_uri": source_uri,
            })
    return pd.DataFrame(all_rows)

# Flatten the batch result from Test 5
chunks_pandas = flatten_prep_search_response(result5)
print(f"Flattened {len(result5)} documents into {len(chunks_pandas)} chunk rows")
display(chunks_pandas)

In [0]:
# ── Flatten JSON response using Spark SQL + VARIANT ──────────────────
# This mirrors the official ai_prep_search documentation pattern:
#   LATERAL variant_explode(result:document.contents)
# Works for both direct model output and endpoint responses.

from pyspark.sql import functions as F

# Create a Spark DataFrame from the batch results
response_rows = [(resp,) for resp in result5["response"].tolist()]
model_output_df = spark.createDataFrame(response_rows, schema=["response_json"])
model_output_df.createOrReplaceTempView("model_output")

# ── variant_explode approach (matches official ai_prep_search docs) ──
chunks_spark = spark.sql("""
SELECT
    chunk.value:chunk_id::STRING         AS chunk_id,
    chunk.value:chunk_position::INT      AS chunk_position,
    chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
    chunk.value:chunk_to_embed::STRING   AS chunk_to_embed,
    chunk.value:metadata                 AS metadata,
    parse_json(response_json):document.source_uri::STRING AS source_uri
FROM
    model_output,
    LATERAL variant_explode(
        parse_json(response_json):document.contents
    ) AS chunk
""")

print(f"Spark DataFrame: {chunks_spark.count()} chunk rows")
display(chunks_spark)

In [0]:
# ── Write flattened chunks to a Delta table ──────────────────────────
# Uncomment and run to persist chunks for use as a Vector Search source.
# Use chunk_to_embed as the embedding column and chunk_id as primary key.

# TARGET_TABLE = f"{CATALOG}.{SCHEMA}.prep_search_chunks"
#
# chunks_spark.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
# print(f"Wrote {chunks_spark.count()} rows to {TARGET_TABLE}")
#
# -- Or via SQL after the variant_explode query above:
# spark.sql(f"""
# CREATE OR REPLACE TABLE {TARGET_TABLE} AS
# SELECT
#     chunk.value:chunk_id::STRING         AS chunk_id,
#     chunk.value:chunk_position::INT      AS chunk_position,
#     chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
#     chunk.value:chunk_to_embed::STRING   AS chunk_to_embed,
#     chunk.value:metadata                 AS metadata,
#     parse_json(response_json):document.source_uri::STRING AS source_uri
# FROM
#     model_output,
#     LATERAL variant_explode(
#         parse_json(response_json):document.contents
#     ) AS chunk
# """)

print("Uncomment the block above to persist chunks to Delta.")
print(f"Target table: {CATALOG}.{SCHEMA}.prep_search_chunks")

## End-to-End Pipeline: Parse → Prep → Search

Chain `ai-parse-document-gemini` and `ai-prep-search-fmapi` to go from a raw document to search-ready chunks in one pass.

1. **Python SDK** — single-document walkthrough via `serving_endpoints.query`
2. **SQL `ai_query`** — batch-ready pipeline that processes a Volume of documents in one statement

In [0]:
# ── End-to-End Chain: parse-document → prep-search → flat table ──────
# Calls both serving endpoints in sequence on a single document.

import base64
from pathlib import Path

ws = WorkspaceClient()

# Pick a sample document from the parse docs volume
DOC_PATH = f"/Volumes/{CATALOG}/bio_track/unstructured/20260618 - Meal on plane.png"
doc_bytes = Path(DOC_PATH).read_bytes()
print(f"Document: {Path(DOC_PATH).name} ({len(doc_bytes):,} bytes)")

# ── Step 1: Parse document → ai-parse-document-gemini ────────────────
parse_resp = ws.serving_endpoints.query(
    name="ai-parse-document-gemini",
    dataframe_records=[{
        "file_b64":                   base64.b64encode(doc_bytes).decode(),
        "mime_type":                  "image/png",
        "file_path":                  DOC_PATH,
        "file_name":                  Path(DOC_PATH).name,
        "file_size":                  len(doc_bytes),
        "file_modification_time":     "2026-06-18T00:00:00Z",
        "page_limit":                 1,
        "page_index":                 -1,
        "reformat":                   True,
    }],
)
parsed_json = parse_resp.predictions[0]["response"]
parsed_envelope = json.loads(parsed_json)
n_elements = len(parsed_envelope.get("document", {}).get("elements", []))
print(f"\n\u2713 Step 1: ai-parse-document-gemini \u2192 "
      f"{n_elements} elements, {len(parsed_json):,} chars")

# ── Step 2: Chunk & enrich → ai-prep-search-fmapi ───────────────────
prep_resp = ws.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{
        "parsed_doc_json":  parsed_json,
        "source_uri":       DOC_PATH,
    }],
)
prep_json = prep_resp.predictions[0]["response"]
prep_envelope = json.loads(prep_json)
n_chunks = len(prep_envelope["document"]["contents"])
print(f"\u2713 Step 2: {ENDPOINT_NAME} \u2192 {n_chunks} search-ready chunks")

# ── Step 3: Flatten into target table schema ───────────────────────
rows = []
source_uri = prep_envelope["document"]["source_uri"]
for chunk in prep_envelope["document"]["contents"]:
    rows.append({
        "chunk_id":          chunk["chunk_id"],
        "chunk_position":    chunk["chunk_position"],
        "chunk_to_retrieve": chunk["chunk_to_retrieve"],
        "chunk_to_embed":    chunk["chunk_to_embed"],
        "metadata":          json.dumps(chunk["metadata"]),
        "source_uri":        source_uri,
    })

chain_df = pd.DataFrame(rows)
print(f"\n\u2713 End-to-end: 1 document \u2192 {n_elements} elements "
      f"\u2192 {n_chunks} chunks\n")
display(chain_df)

In [0]:
%sql
-- ── Batch Pipeline: ai_query chain over a Volume of documents ───────
-- Replace the read_files path to process your own documents.
-- Writes directly to a Delta table usable as a Vector Search source.

WITH parsed_documents AS (
  SELECT
    path,
    ai_query(
      'ai-parse-document-gemini',
      named_struct(
        'file_b64',                   base64(content),
        'mime_type',                  CASE
                                        WHEN path LIKE '%.pdf'  THEN 'application/pdf'
                                        WHEN path LIKE '%.png'  THEN 'image/png'
                                        WHEN path LIKE '%.jpg'
                                          OR path LIKE '%.jpeg' THEN 'image/jpeg'
                                      END,
        'file_path',                  path,
        'file_name',                  regexp_extract(path, '([^/]+)$'),
        'file_size',                  CAST(length(content) AS BIGINT),
        'file_modification_time',     '2026-01-01T00:00:00Z',
        'page_limit',                 CAST(4 AS BIGINT),
        'page_index',                 CAST(-1 AS BIGINT),
        'reformat',                   true
      ),
      returnType => 'STRUCT<response: STRING>'
    ) AS parse_result
  FROM read_files(
    '/Volumes/justinm_demo/bio_track/unstructured/20260618 - Meal on plane.png',
    format => 'binaryFile'
  )
),
prepped_documents AS (
  SELECT
    path,
    ai_query(
      'ai-prep-search-fmapi',
      named_struct(
        'parsed_doc_json',  parse_result.response,
        'source_uri',       path
      ),
      returnType => 'STRUCT<response: STRING>'
    ) AS prep_result
  FROM parsed_documents
)
SELECT
  chunk.value:chunk_id::STRING          AS chunk_id,
  chunk.value:chunk_position::INT       AS chunk_position,
  chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
  chunk.value:chunk_to_embed::STRING    AS chunk_to_embed,
  chunk.value:metadata                  AS metadata,
  prepped_documents.path                AS source_uri
FROM
  prepped_documents,
  LATERAL variant_explode(
    parse_json(prep_result.response):document.contents
  ) AS chunk

## Built-in vs Custom `ai_prep_search` Comparison

Side-by-side comparison using the **same parsed document** as input:

| Method | Description |
| --- | --- |
| **Built-in** `ai_prep_search()` | Databricks native SQL AI function |
| **Custom** `ai-prep-search-fmapi` | This notebook's MLflow endpoint (Gemini 3.8 Flash via FMAPI) |

Each pipeline uses its own parser because the built-in `ai_prep_search()` rejects custom parser output (`"The input content is invalid."`). This is a true **end-to-end** comparison of both full pipelines.

| Pipeline | Parser | Prep/Chunker |
| --- | --- | --- |
| **Built-in** | `ai_parse_document()` | `ai_prep_search()` |
| **Custom** | `ai-parse-document-fmapi` endpoint | `ai-prep-search-fmapi` endpoint |

In [0]:
import json
import pandas as pd
from difflib import SequenceMatcher

DOC_PATH = '/Volumes/justinm_demo/bio_track/unstructured/20260618 - Meal on plane.png'
DOC_PATH = "/Volumes/justinm_demo/bio_track/unstructured/cynthia-data-1-rupert-braun.pdf"

# ═══════════════════════════════════════════════════════════════════════
# Step 1a — Parse with BUILT-IN ai_parse_document (for ai_prep_search)
# ═══════════════════════════════════════════════════════════════════════
print("Step 1a: Parsing with built-in ai_parse_document()...")
builtin_parsed = spark.sql(f"""
  SELECT path, CAST(ai_parse_document(content) AS STRING) AS parsed_json
  FROM read_files('{DOC_PATH}', format => 'binaryFile')
""").toPandas()
spark.createDataFrame(builtin_parsed).createOrReplaceTempView("_builtin_parsed")
print("✓ Built-in parser done\n")

# ═══════════════════════════════════════════════════════════════════════
# Step 1b — Parse with ai-parse-document-fmapi (for custom endpoint)
# ═══════════════════════════════════════════════════════════════════════
print("Step 1b: Parsing with ai-parse-document-fmapi endpoint...")
custom_parsed = spark.sql(f"""
  SELECT
    path,
    ai_query(
      'ai-parse-document-fmapi',
      named_struct(
        'file_b64',                   base64(content),
        'mime_type',                  'image/png',
        'file_path',                  path,
        'file_name',                  regexp_extract(path, '([^/]+)$'),
        'file_size',                  CAST(length(content) AS BIGINT),
        'file_modification_time',     '2026-01-01T00:00:00Z',
        'page_limit',                 CAST(4 AS BIGINT),
        'page_index',                 CAST(-1 AS BIGINT),
        'reformat',                   true
      ),
      returnType => 'STRUCT<response: STRING>'
    ).response AS parsed_json
  FROM read_files('{DOC_PATH}', format => 'binaryFile')
""").toPandas()
spark.createDataFrame(custom_parsed).createOrReplaceTempView("_custom_parsed")
print("✓ FMAPI parser done\n")

# ═══════════════════════════════════════════════════════════════════════
# Step 2 — Built-in ai_prep_search() on built-in parser output
# ═══════════════════════════════════════════════════════════════════════
print("Step 2: Running built-in ai_prep_search()...")
builtin = spark.sql("""
  WITH prepped AS (
    SELECT ai_prep_search(parse_json(parsed_json)) AS result FROM _builtin_parsed
  )
  SELECT
    chunk.value:chunk_id::STRING          AS chunk_id,
    chunk.value:chunk_position::INT       AS chunk_position,
    chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
    chunk.value:chunk_to_embed::STRING    AS chunk_to_embed,
    chunk.value:metadata                  AS metadata,
    chunk.value:pages                     AS pages,
    result:document.source_uri::STRING    AS source_uri
  FROM prepped, LATERAL variant_explode(result:document.contents) AS chunk
  ORDER BY 1
""").toPandas()
print(f"✓ Built-in: {len(builtin)} chunks\n")

# ═══════════════════════════════════════════════════════════════════════
# Step 3 — Custom ai-prep-search-fmapi on FMAPI parser output
# ═══════════════════════════════════════════════════════════════════════
print("Step 3: Running custom ai-prep-search-fmapi endpoint...")
custom = spark.sql("""
  WITH prepped AS (
    SELECT ai_query(
      'ai-prep-search-fmapi',
      named_struct(
        'parsed_doc_json', parsed_json,
        'source_uri',      path
      ),
      returnType => 'STRUCT<response: STRING>'
    ) AS result
    FROM _custom_parsed
  )
  SELECT
    chunk.value:chunk_id::STRING          AS chunk_id,
    chunk.value:chunk_position::INT       AS chunk_position,
    chunk.value:chunk_to_retrieve::STRING AS chunk_to_retrieve,
    chunk.value:chunk_to_embed::STRING    AS chunk_to_embed,
    chunk.value:metadata                  AS metadata,
    chunk.value:pages                     AS pages,
    parse_json(result.response):document.source_uri::STRING AS source_uri
  FROM prepped,
    LATERAL variant_explode(parse_json(result.response):document.contents) AS chunk
  ORDER BY 1
""").toPandas()
print(f"✓ Custom:  {len(custom)} chunks\n")

# ═══════════════════════════════════════════════════════════════════════
# Summary stats
# ═══════════════════════════════════════════════════════════════════════
print("═" * 70)
print("  SUMMARY")
print("═" * 70)

def _stats(df):
    r = df["chunk_to_retrieve"].str.len()
    e = df["chunk_to_embed"].str.len()
    return [len(df), int(r.mean()), int(e.mean()), int(r.sum())]

summary = pd.DataFrame({
    "Metric": [
        "Chunk count",
        "Avg retrieve length (chars)",
        "Avg embed length (chars)",
        "Total retrieve chars",
    ],
    "Built-in ai_prep_search": _stats(builtin) if len(builtin) else [0]*4,
    "Custom FMAPI endpoint":   _stats(custom)  if len(custom)  else [0]*4,
})
display(summary)

# ═══════════════════════════════════════════════════════════════════════
# Side-by-side chunk_to_retrieve
# ═══════════════════════════════════════════════════════════════════════
max_pos = max(
    builtin["chunk_position"].max() if len(builtin) else -1,
    custom["chunk_position"].max()  if len(custom)  else -1,
)

rows = []
for pos in range(int(max_pos) + 1):
    b = builtin.loc[builtin["chunk_position"] == pos, "chunk_to_retrieve"]
    c = custom.loc[custom["chunk_position"] == pos, "chunk_to_retrieve"]
    b_text = b.iloc[0] if len(b) else ""
    c_text = c.iloc[0] if len(c) else ""
    sim = SequenceMatcher(None, b_text, c_text).ratio() if (b_text and c_text) else 0.0
    trunc = lambda s: (s[:150] + "…") if len(s) > 150 else s
    rows.append({
        "Pos": pos,
        "Built-in chunk_to_retrieve": trunc(b_text),
        "Custom chunk_to_retrieve":   trunc(c_text),
        "Similarity": f"{sim:.0%}",
    })

print("\n\nchunk_to_retrieve — side-by-side (truncated at 150 chars):")
display(pd.DataFrame(rows))

# ═══════════════════════════════════════════════════════════════════════
# Detailed chunk_to_embed for each matching position
# ═══════════════════════════════════════════════════════════════════════
for pos in range(min(int(max_pos) + 1, 3)):   # show first 3 chunks
    print(f"\n{'═' * 70}")
    print(f"  chunk_to_embed — Position {pos}")
    print("═" * 70)
    b = builtin.loc[builtin["chunk_position"] == pos, "chunk_to_embed"]
    c = custom.loc[custom["chunk_position"] == pos, "chunk_to_embed"]
    if len(b):
        print(f"\n┌─── Built-in ai_prep_search ───")
        print(b.iloc[0])
        print("└" + "─" * 40)
    if len(c):
        print(f"\n┌─── Custom FMAPI endpoint ───")
        print(c.iloc[0])
        print("└" + "─" * 40)

In [0]:
display(custom)

In [0]:
display(builtin)